# County Obesity Hotspot Analysis Workflow (Multi data Download)

This notebook demonstrates a clean multi-agent GAS workflow for a real public-health use case:

**Where are the statistically significant clusters (hot spots and cold spots) of adult obesity prevalence across U.S. counties?**


## Install GAS Client SDK


In [1]:
%pip install -q gas-client

## Imports

In [2]:
from urllib.parse import urljoin
from IPython.display import HTML, Image, display
from gas_client import GasClient

## User Settings


In [3]:

server_url = "http://127.0.0.1:4042"


openai_api_key = "YOUR_OPENAI_API_KEY"
timeout_seconds = 1800

client = GasClient(
    server_url,
    openai_api_key=openai_api_key,
    artifact_delivery="URL",
)

credentials = {"OPENAI_API_KEY": openai_api_key}

## Bind to the Five Agents

In [4]:
data_agent = client.agent("geospatial_data_retrieval_agent")
vector_agent = client.agent("vector_analysis_agent")
projection_agent = client.agent("map_projection_agent")
stats_agent = client.agent("spatial_statistics_agent")
mapping_agent = client.agent("mapping_agent")

for agent in [data_agent, vector_agent, projection_agent, stats_agent, mapping_agent]:
    print(agent.agent_id, agent.status().get("status"))

geospatial_data_retrieval_agent available
vector_analysis_agent available
map_projection_agent available
spatial_statistics_agent available
mapping_agent available


## Helper Functions


In [5]:
def absolute_url(url):
    if url.startswith("/"):
        return urljoin(server_url, url)
    return url


def first_artifact_url(task_result, preferred_extensions=None):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    preferred_extensions = preferred_extensions or []

    for extension in preferred_extensions:
        for artifact in artifacts:
            url = artifact.get("url")
            filename = artifact.get("filename") or artifact.get("name") or url or ""
            if url and str(filename).lower().endswith(extension.lower()):
                return absolute_url(url)

    for artifact in artifacts:
        if artifact.get("url"):
            return absolute_url(artifact["url"])

    raise RuntimeError("The task returned no artifact URL.")


def display_visual_artifacts(task_result):
    artifacts = task_result.get("outputs", {}).get("artifacts", [])
    for artifact in artifacts:
        url = artifact.get("url")
        filename = artifact.get("filename") or artifact.get("name") or ""
        if not url:
            continue

        display_url = absolute_url(url)
        lower_name = str(filename).lower()
        if lower_name.endswith(".png"):
            display(Image(url=display_url))
        elif lower_name.endswith(".html"):
            display(HTML(f'<a href="{display_url}" target="_blank">Open HTML artifact: {filename}</a>'))
        else:
            display(HTML(f'<a href="{display_url}" target="_blank">Open artifact: {filename}</a>'))


def run_streaming_task(agent, instructions, *, input_datasets=None, title=None):
    if title:
        print("\n" + "=" * 80)
        print(title)
        print("=" * 80)

    final_result = None
    for event in agent.execute_task(
        instructions,
        mode="stream",
        input_datasets=input_datasets,
        artifact_delivery="URL",
        credentials=credentials,
        timeout=timeout_seconds,
    ):
        client.print_stream_event(event)
        if event.get("event") == "task_result":
            final_result = event.get("payload")

    if final_result is None:
        raise RuntimeError("The stream ended before returning a task_result event.")

    client.print_task_summary(final_result)
    return final_result

## Step 1: Multi-Download (Counties + CDC Obesity Prevalence)


In [6]:
multi_retrieval_result = run_streaming_task(
    data_agent,
    (
        "Please download two datasets for the contiguous United States:\n"
        "1) Use the U.S. Census Bureau TIGER boundary source. Download 2020 county polygons "
        "for the contiguous United States. Exclude Alaska, Hawaii, Puerto Rico, and other "
        "territories. Return one clean GeoPackage with county geometry, GEOID, county name, "
        "state name or abbreviation, STATEFP, and COUNTYFP fields.\n"
        "2) Use the CDC PLACES data source. Download 2020 county-level crude prevalence of "
        "adult obesity for the contiguous United States. Return one clean dataset with county "
        "GEOID/FIPS, county name, state identifier, year, measure name, and the obesity "
        "prevalence value as a numeric column. Geometry is not required for this dataset.\n\n"
        "No source API key is required for either dataset."
    ),
    title="Multi-Download: County Boundaries + CDC Obesity Prevalence",
)

# Collect every artifact URL the multi-download produced, in sub-request order.
downloaded_urls = [
    absolute_url(a["url"])
    for a in (multi_retrieval_result.get("outputs", {}).get("artifacts", []) or [])
    if a.get("url")
]

if len(downloaded_urls) < 2:
    raise RuntimeError(
        f"Expected 2 downloaded artifacts (counties, obesity) but got {len(downloaded_urls)}. "
        "Inspect the multi-retrieval task output above before proceeding.\n"
        f"Artifacts: {downloaded_urls}"
    )

# Position-based assignment matches the order in the multi-download prompt.
county_boundary_url, obesity_url = downloaded_urls[0], downloaded_urls[1]

print("counties:", county_boundary_url)
print("obesity :", obesity_url)

sub_tasks = multi_retrieval_result.get("outputs", {}).get("sub_tasks", []) or []
if sub_tasks:
    print("\nSub-task breakdown (verify the request-to-artifact mapping):")
    for record in sub_tasks:
        req_preview = (record.get("sub_request") or "")[:90]
        print(
            f"  #{record.get('sub_request_index')} "
            f"({record.get('artifact_count')} artifact(s)): "
            f"{req_preview}..."
        )

## Step 2: Join Boundaries With Obesity Prevalence

In [7]:
join_result = run_streaming_task(
    vector_agent,
    (
        "Join the contiguous US county boundary dataset with the 2020 CDC PLACES county-level adult "
        "obesity prevalence dataset. Use GEOID or county FIPS as the join key (pad to 5 characters if "
        "needed so STATEFP + COUNTYFP matches the FIPS string). Preserve county boundary geometry and "
        "key identifiers. Rename the obesity prevalence column to a clean numeric field named "
        "'obesity_rate'. Keep county name, state, GEOID, year, and measure fields when available. "
        "Drop counties with missing obesity_rate. Return one GeoPackage ready for spatial statistics."
    ),
    input_datasets=[county_boundary_url, obesity_url],
    title="Join County Boundaries With CDC Obesity Prevalence",
)

county_obesity_url = first_artifact_url(
    join_result, preferred_extensions=[".gpkg", ".geojson", ".json"]
)
county_obesity_url


Join County Boundaries With CDC Obesity Prevalence
[21:57:39] stream_connected: Streaming connection established.
[21:57:39] Vector Analysis Agent: I received your request.
[21:57:39] Vector Analysis Agent: I need to inspect the request text and any provided dataset references before running the agent. I found 2 dataset reference(s).
[21:57:40] Vector Analysis Agent: I found the required credentials and can start the model-backed workflow.
[21:57:40] task_accepted: Task accepted. Starting streaming execution.
[21:57:40] Vector Analysis Agent: Next I will run the agent with the prepared inputs.
[21:57:40] Vector Analysis Agent: I will load the requested vector/tabular inputs, run code-driven analysis, and save a final dataset artifact from 2 dataset reference(s).
[21:57:40] Vector Analysis Agent: I detected a common vector operation and will first try a deterministic GeoPandas workflow.
[21:57:40] Vector Analysis Agent: Joined 3108 spatial features with 3109 table/vector records using 

'http://127.0.0.1:4042/agents/vector_analysis_agent/data/vector_analysis_agent-9780-otjp-5545.gpkg'

## Step 3: Reproject to EPSG:5070 (Conus Albers)


In [8]:
projection_result = run_streaming_task(
    projection_agent,
    (
        "Reproject the input county dataset from its current CRS to EPSG:5070 "
        "(NAD83 / Conus Albers Equal Area). Preserve every attribute (including GEOID, county "
        "name, state, year, measure, and the numeric obesity_rate field). Return one GeoPackage "
        "in EPSG:5070 ready for downstream spatial statistics and mapping."
    ),
    input_datasets=[county_obesity_url],
    title="Reproject Joined Dataset to EPSG:5070",
)

county_obesity_projected_url = first_artifact_url(
    projection_result, preferred_extensions=[".gpkg", ".geojson", ".json"]
)
county_obesity_projected_url


Reproject Joined Dataset to EPSG:5070
[21:57:41] stream_connected: Streaming connection established.
[21:57:41] Map Projection Agent: I received your request.
[21:57:41] Map Projection Agent: I need to inspect the request text and any provided dataset references before running the agent. I found 1 dataset reference(s).
[21:57:41] Map Projection Agent: I found the required credentials and can start the model-backed workflow.
[21:57:42] task_accepted: Task accepted. Starting streaming execution.
[21:57:42] Map Projection Agent: Next I will run the agent with the prepared inputs.
[21:57:42] Map Projection Agent: I will inspect 1 dataset reference(s), identify their current CRS, and choose an appropriate target projection for the request.
[21:57:42] Map Projection Agent: I am loading an input dataset so I can inspect its current coordinate reference system.
[21:57:42] Map Projection Agent: I am selecting a target CRS from the request text and dataset extent using local pyproj/geopandas lo

'http://127.0.0.1:4042/agents/map_projection_agent/data/map_projection_agent-9144-qprm-4872.gpkg'

## Step 4: Local Moran's I (LISA) Cluster Analysis


In [ ]:
county_obesity_projected_url = "http://127.0.0.1:4042/agents/map_projection_agent/data/map_projection_agent-9144-qprm-4872.gpkg"
lisa_result = run_streaming_task(
    stats_agent,
    (
        "Perform an Exploratory Spatial Data Analysis on the input county dataset using PySAL. "
        "The input is already projected to EPSG:5070; do not reproject again. "
        "Target variable: obesity_rate. "
        "1) Build Queen-contiguity spatial weights and row-standardize them. "
        "2) Compute the Global Moran's I statistic with 999 permutations and report the I value, "
        "expected I, z-score, and pseudo p-value. "
        "3) Compute Local Moran's I (LISA) with 999 permutations and classify each county into "
        "'High-High', 'Low-Low', 'High-Low', 'Low-High', or 'Not Significant' at p < 0.05. "
        "4) Return a polished HTML report with the Moran scatterplot, a LISA significance map, and "
        "a LISA cluster map embedded. "
        "5) **REQUIRED**: also return a GeoPackage (.gpkg) primary artifact containing the input "
        "county polygons augmented with the new fields 'local_I', 'local_p_value', and "
        "'lisa_cluster' (the categorical label). This GeoPackage is required for the next mapping "
        "step. "
        "Interpret the results in the report in plain English suitable for a public-health audience."
    ),
    input_datasets=[county_obesity_projected_url],
    title="Local Moran's I (LISA) Hotspot Analysis",
)

lisa_cluster_url = first_artifact_url(
    lisa_result, preferred_extensions=[".gpkg", ".geojson", ".json"]
)

lisa_cluster_url

### View the Statistical Report


In [ ]:
display_visual_artifacts(lisa_result)